# Reproducing Table 1 of *Scalable k-Means Clustering via Lightweight Coresets*

Bachem, Lucic & Krause, KDD 2018 ([arXiv:1702.08248](https://arxiv.org/abs/1702.08248)).

This notebook runs the same experiment as `reproduce_table1.py`, step by step:

1. load a dataset,
2. build a coreset of size m with Uniform sampling, the lightweight coreset (LWCS, Algorithm 1) and the sensitivity coreset (CS),
3. run weighted k-means++ on the coreset,
4. measure the cost of those centres on the **full** data and compare with k-means++ on the full data.

Set `DATASET` below. `synthetic` runs anywhere in a few minutes; the real datasets need `datasets/download_datasets.py` first.

In [ ]:
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from coresets import kmeans_cost, lightweight_coreset, sensitivity_coreset, uniform_sample
from reproduce_table1 import load_dataset, run_kmeans

DATASET = "synthetic"      # synthetic | kdd | song | rna
K_VALUES = [100]           # paper: [100, 500]
M_VALUES = [1000, 2000, 5000]
METHODS = ["uniform", "lwcs", "cs"]
REPS = 5                   # paper: 50
FULL_REPS = 3

## 1. Load the data

In [ ]:
X = load_dataset(DATASET, None)
print(f"{DATASET}: n = {X.shape[0]:,} rows, d = {X.shape[1]} columns")

## 2. Look at one lightweight coreset

The sampling probability is q(x) = 1/2 · 1/n + 1/2 · d(x, μ)² / Σ d(x', μ)². Points far from the mean get picked more often but carry smaller weights, and on average the weights add up to n.

In [ ]:
C, w = lightweight_coreset(X, 1000, np.random.default_rng(0))
print(f"coreset: {C.shape[0]} rows, weights sum to {w.sum():,.0f} (n = {X.shape[0]:,})")
print(f"smallest weight {w.min():.1f}, largest weight {w.max():.1f}")

## 3. Run the experiment

In [ ]:
def build(method, m, k, rng):
    if method == "uniform":
        return uniform_sample(X, m, rng)
    if method == "lwcs":
        return lightweight_coreset(X, m, rng)
    return sensitivity_coreset(X, m, k, rng)

rows = []
for k in K_VALUES:
    full_costs, full_times = [], []
    for r in range(FULL_REPS):
        t0 = time.perf_counter()
        Q = run_kmeans(X, k, seed=r)
        full_times.append(time.perf_counter() - t0)
        full_costs.append(kmeans_cost(X, Q))
    full_cost, full_time = np.mean(full_costs), np.mean(full_times)
    print(f"k={k}: full k-means++ takes {full_time:.1f} s")
    for m in M_VALUES:
        for method in METHODS:
            for r in range(REPS):
                t0 = time.perf_counter()
                C, w = build(method, m, k, np.random.default_rng(1000 + r))
                Q = run_kmeans(C, k, seed=r, weights=w)
                elapsed = time.perf_counter() - t0
                rows.append(dict(k=k, m=m, method=method, rep=r,
                                 rel_error=(kmeans_cost(X, Q) - full_cost) / full_cost,
                                 speedup=full_time / elapsed))
df = pd.DataFrame(rows)

## 4. Results in the layout of Table 1 (mean over repetitions, ± 95% confidence interval)

In [ ]:
def ci(x):
    return 1.96 * x.std(ddof=1) / np.sqrt(len(x))

table = df.groupby(["k", "m", "method"]).agg(
    error_pct=("rel_error", lambda x: 100 * x.mean()),
    ci_pct=("rel_error", lambda x: 100 * ci(x)),
    speedup=("speedup", "mean"),
).round(1)
table

In [ ]:
fig, axes = plt.subplots(1, len(K_VALUES), figsize=(6 * len(K_VALUES), 4), squeeze=False)
for ax, k in zip(axes[0], K_VALUES):
    sub = table.loc[k].reset_index()
    for method, g in sub.groupby("method"):
        ax.errorbar(g["m"], g["error_pct"], yerr=g["ci_pct"], marker="o", capsize=3, label=method)
    ax.set_xscale("log"); ax.set_yscale("log")
    ax.set_xlabel("coreset size m"); ax.set_ylabel("relative error (%)")
    ax.set_title(f"{DATASET}, k = {k}")
    ax.legend()
plt.tight_layout()

## 5. Compare with the paper

Fill in the paper's numbers from Table 1 for the same dataset and k, and discuss the differences in the report (see the README for likely reasons: hardware and library speed, solver settings, number of repetitions, preprocessing).